In [ ]:
import pandas as pd
import numpy as np

'''
데이터 로드 + `제거
'''
def load_csv(txt_dir):
    df = pd.read_csv('E:/B068. 서울시 연립 다세대 임대 예측시세/2. 파일데이터/'+ txt_dir, sep='|', dtype = str, engine='python')
    df.columns = [str(c).strip().strip("`") for c in df.columns]
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.strip("`")
    return df

In [ ]:
df_주택기본정보 = load_csv('1. 주택기본정보/jtbasicinfo_202208.txt')
df_전세예측 = load_csv('3. 전세임대 예측시세/depo_hosiseinfo_202208.txt')
df_월세예측 = load_csv('5. 월세임대 예측시세/rent_hosiseinfo_202208.txt')

In [ ]:
ref_year = 2022
LOW_Q = 0.2
old_threshold = 30

def prepare_price_ratio(df, price_col, dong_code_col, label):
    '''
    전세/월세 각각에 대해
    하위 분위 기준 계산
    저가여부 확인자 생성
    행정동별 저가 비율 및 건수 계산
    '''
    tmp = df.copy()
    tmp = tmp[tmp[price_col].notna()].copy()
    q = tmp[price_col].quantile(LOW_Q)
    # 저가 indicator
    tmp[f'is_low_{label}'] = (tmp[price_col] <= q).astype(int)

    # 동별 집계
    agg = (tmp.groupby(dong_code_col, dropna=False).agg(n_obs=(price_col, 'size'), low_ratio=(f'is_low_{label}', 'mean')).reset_index())

    agg = agg.rename(columns={'n_obs':f'n_{label}', 'low_ratio':f'low_ratio_{label}'})
    return agg, q


def prepare_old_ratio(df, build_year_col, dong_code_col):
    '''
    건축연도 기준 노후 비율 계산
    '''
    tmp = df.copy()
    tmp['SYDATE_parsed']= pd.to_datetime(tmp['SYDATE'], errors='coerce')
    tmp['build_year_final']=tmp['SYDATE_parsed'].dt.year
    
    tmp = tmp[tmp['build_year_final'].notna()].copy()
    tmp.loc[tmp['build_year_final']==0, 'build_year_final']=np.nan
    tmp = tmp[tmp['build_year_final'].notna()].copy()
    tmp['building_age'] = ref_year - tmp['build_year_final'].astype(int)
    tmp['is_old'] = (tmp['building_age'] >= old_threshold).astype(int)

    agg = (tmp.groupby(dong_code_col, dropna=False).agg(n_building=('building_age', 'size'), old_ratio=('is_old', 'mean')).reset_index())

    return agg

In [ ]:
'''
PNU코드의 앞 10자리와 뒤에 법정동 코드까지 이어지는 10자리 합이 같은지 체크 -> 동일
'''
df_주택기본정보['PNU'] = df_주택기본정보['PNU'].astype(str).str.strip()
df_주택기본정보['bjdong_code'] = df_주택기본정보['PNU'].str[:10]
df_주택기본정보['SREG'] = df_주택기본정보['SREG'].astype(str).str.strip().str.zfill(5)
df_주택기본정보['SEUB'] = df_주택기본정보['SEUB'].astype(str).str.strip().str.zfill(5)
df_주택기본정보['bjdong_code_check'] = df_주택기본정보['SREG']+df_주택기본정보['SEUB']
mismatch = df_주택기본정보[df_주택기본정보['bjdong_code'] != df_주택기본정보['bjdong_code_check']]
print(len(mismatch))

In [ ]:
'''
법정동 코드 추출
'''
df_주택기본정보['bjdong_code'] = df_주택기본정보['PNU'].str[:10]
df_전세예측['bjdong_code'] = df_전세예측['PNU'].str[:10]
df_월세예측['bjdong_code'] = df_월세예측['PNU'].str[:10]

dong_code_col = 'bjdong_code'
price_col_depo = 'DEPO_PRED'
price_col_rent = 'PRED_RENT'
build_year_col = 'SYDATE'

In [ ]:
df_주택기본정보['BUILDYEAR']  = pd.to_numeric(df_주택기본정보['BUILDYEAR'], errors='coerce')
df_전세예측[price_col_depo] = pd.to_numeric(df_전세예측[price_col_depo], errors='coerce')
df_월세예측[price_col_rent] = pd.to_numeric(df_월세예측[price_col_rent], errors='coerce')

In [ ]:
depo_agg, depo_q = prepare_price_ratio(df_전세예측, price_col_depo, dong_code_col, label='depo')
rent_agg, rent_q = prepare_price_ratio(df_월세예측, price_col_rent, dong_code_col, label='rent')
old_agg = prepare_old_ratio(df_주택기본정보, build_year_col, dong_code_col)

In [ ]:
hvi = pd.merge(depo_agg, rent_agg, on=[dong_code_col], how='outer')
hvi['n_depo']=hvi['n_depo'].fillna(0)
hvi['n_rent']=hvi['n_rent'].fillna(0)
hvi['low_ratio_depo'] = hvi['low_ratio_depo'].fillna(np.nan)
hvi['low_ratio_rent'] = hvi['low_ratio_rent'].fillna(np.nan)

In [ ]:
def weighted_low_ratio(row):
    n_depo = row['n_depo']
    n_rent = row['n_rent']
    total = n_depo + n_rent
    if total == 0: return np.nan
    val = 0
    if pd.notna(row['low_ratio_depo']):
        val+= n_depo*row['low_ratio_depo']
    if pd.notna(row['low_ratio_rent']):
        val += n_rent*row['low_ratio_rent']
    return val/total

hvi['low_ratio_combined'] = hvi.apply(weighted_low_ratio, axis =1)

In [ ]:
hvi = pd.merge(hvi, old_agg, on=dong_code_col, how='left')

In [ ]:
df_전세예측['bjdong_code'].value_counts()

In [ ]:
tmp2 = df_주택기본정보.copy()
tmp2['buildyear_num'] = pd.to_numeric(tmp2['BUILDYEAR'], errors = 'coerce')
tmp2.loc[tmp2['buildyear_num']==0, 'buildyear_num'] = np.nan
tmp2['sydate_parsed'] = pd.to_datetime(tmp2['SYDATE'], errors='coerce')
print(tmp2['sydate_parsed'].notna().mean())

In [ ]:
tmp2['buildyear_final']=tmp2['sydate_parsed'].dt.year
tmp2['buildyear_final'].describe()

In [ ]:
def zscore(series): 
    return (series- series.mean())/ series.std(ddof=0)

hvi['z_low'] = zscore(hvi['low_ratio_combined'])
hvi['z_old'] = zscore(hvi['old_ratio'])

hvi['HVI_score_z'] = 0.5 * hvi['z_low'] + 0.5*hvi['z_old']


In [ ]:
hvi['score_low'] = pd.qcut(hvi['low_ratio_combined'], q=5, labels=[1,2,3,4,5]).astype(float)
hvi['score_old'] = pd.qcut(hvi['old_ratio'], q=5, labels=[1,2,3,4,5]).astype(float)

hvi['HVI_score_sum'] = hvi['score_low']+hvi['score_old']
hvi['HVI_rank']=hvi['HVI_score_sum'].rank(ascending=False, method='min')
hvi['HVI_grade'] = pd.qcut(hvi['HVI_score_sum'], q=5, labels=['매우낮음','낮음','보통','높음','매우높음'])


In [ ]:
result = hvi[[dong_code_col, 'n_depo', 'n_rent', 'low_ratio_depo','low_ratio_rent','low_ratio_combined','n_building','old_ratio','HVI_score_z','HVI_score_sum','HVI_rank','HVI_grade']].copy()
print(result.head())
print('전세 하위 분위 기준값: ', depo_q)
print('월세 하위 분위 기준값: ', rent_q)

In [ ]:
'''
validation
'''
print(result[['n_depo','n_rent','n_building']].describe())

In [ ]:
print(result.isna().mean().sort_values(ascending=False))

In [ ]:
print(result[['HVI_score_z','HVI_score_sum']].describe())

In [ ]:
import matplotlib.pyplot as plt
result['HVI_score_sum'].hist(bins=30)
plt.show()

In [ ]:
print(result.sort_values('HVI_rank').head(20))


In [ ]:
export_hvi = result[[dong_code_col, 'low_ratio_combined','old_ratio','HVI_score_sum','HVI_rank','HVI_grade']].copy()

In [ ]:
export_hvi2 = result[[dong_code_col,'HVI_score_sum','HVI_rank','HVI_grade']].copy()

In [ ]:
export_hvi.to_csv('export_hvi.csv', index=False, encoding='utf-8-sig')
export_hvi2.to_csv('export_hvi2.csv', index=False, encoding='utf-8-sig')